# ErgoPose Risk Classifier — Fine-Tuning and Model Evaluation

This notebook is the **fourth stage** of the *ErgoPose Risk Classifier* project.  
It focuses on fine-tuning the top 3 performing models previously trained in the project. The goal is to optimize their hyperparameters and evaluate their performance to select the best model for deployment.

### Objectives
- Load the previously trained models and their performance metrics.
- Perform a **hyperparameter grid search** to fine-tune the top 3 models.
- Evaluate the models using **5-Fold Cross-Validation**.
- Compare the results in terms of **accuracy**, **precision**, **recall**, and **F1-score**.
- Visualize the performance of the models (e.g., learning curves, accuracy histograms).
- Save the fine-tuned models and performance history for future reference.

### Input and Output
- **Input:**  
  - `models/neural_network.pkl` (Top 3 models)  
  - `data/processed/clean_postural_risk_dataset.csv`
  
- **Outputs:**  
  - `models/fine_tuned_model_1.pkl`  
  - `models/fine_tuned_model_2.pkl`  
  - `models/fine_tuned_model_3.pkl`  
  - Performance history logs (e.g., accuracy, precision, recall, F1-score)  
  - Visualizations (e.g., learning curves, histograms)

### Next Steps
- Fine-tuning the models and analyzing the results will help determine the most effective approach for ergonomic posture classification.

In [1]:
"""
Import necessary libraries for model fine-tuning, evaluation, and visualization.
"""

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
import json
import joblib
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


In [2]:
"""
Configure paths for data, models, and outputs. Initialize device for GPU/CPU usage.
"""

DATA_PATH = Path("../data/processed/clean_postural_risk_dataset.csv")
MODELS_PATH = Path("../models")
OUTPUT_PATH = Path("../outputs")
OUTPUT_PATH.mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

RANDOM_STATE = 42
N_FOLDS = 5
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print(f"Data path: {DATA_PATH}")
print(f"Models directory: {MODELS_PATH}")
print(f"Output directory: {OUTPUT_PATH}")


Using device: cuda
Data path: ../data/processed/clean_postural_risk_dataset.csv
Models directory: ../models
Output directory: ../outputs


In [3]:
"""
Load and verify the preprocessed dataset from previous stages.
Ensure data consistency and prepare for model fine-tuning.
"""

# Load the preprocessed dataset
print("Loading preprocessed dataset...")
data = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {data.shape}")

# Separate features and target
X = data.drop(columns=['upperbody_label'])
y = data['upperbody_label']

# Verify target distribution
print("\nTarget distribution:")
print(y.value_counts())

# Prepare data for PyTorch
X_tensor = torch.FloatTensor(X.values)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_tensor = torch.LongTensor(y_encoded)

print("\nData prepared for training:")
print(f"X tensor shape: {X_tensor.shape}")
print(f"y tensor shape: {y_tensor.shape}")
print(f"Number of classes: {len(label_encoder.classes_)}")


Loading preprocessed dataset...
Dataset shape: (4794, 52)

Target distribution:
upperbody_label
bad     3179
good    1615
Name: count, dtype: int64

Data prepared for training:
X tensor shape: torch.Size([4794, 51])
y tensor shape: torch.Size([4794])
Number of classes: 2


In [4]:
"""
Discover and load previously trained models from the models directory.
Identify all trained models for comprehensive evaluation.
"""

def discover_trained_models(models_path):
    """Discover all trained models and their metadata."""
    models = []
    
    model_files = list(models_path.glob("modelo_*.pth"))
    
    for model_file in model_files:
        metadata_file = model_file.with_suffix('.json')
        
        if metadata_file.exists():
            with open(metadata_file, 'r') as f:
                metadata = json.load(f)
            
            accuracy = float(model_file.stem.split('_')[1])
            
            models.append({
                'file_path': model_file,
                'metadata': metadata,
                'accuracy': accuracy,
                'architecture': f"{metadata.get('hidden_layer_1', 'N/A')}-{metadata.get('hidden_layer_2', 'N/A')}",
                'learning_rate': metadata.get('learning_rate', 'N/A'),
                'activation': metadata.get('funcao_ativacao', 'N/A'),
                'epochs': metadata.get('num_epochs', 'N/A')
            })
    
    return sorted(models, key=lambda x: x['accuracy'], reverse=True)

print("Discovering trained models...")
all_models = discover_trained_models(MODELS_PATH)

print(f"\nFound {len(all_models)} models:")
for i, model_info in enumerate(all_models):
    print(f"{i+1}. Accuracy: {model_info['accuracy']:.3f}, "
          f"Architecture: {model_info['architecture']}, "
          f"LR: {model_info['learning_rate']}, "
          f"Activation: {model_info['activation']}")

model_info_dict = {model['file_path'].name: model for model in all_models}


Discovering trained models...

Found 2 models:
1. Accuracy: 0.700, Architecture: 8-10, LR: 0.0001, Activation: ReLU
2. Accuracy: 0.680, Architecture: 8-10, LR: 0.0001, Activation: ReLU


In [5]:
"""
Evaluate model performance using cross-validation with multiple metrics.
"""

def evaluate_model_performance(model_path, X_data, y_data, n_folds=3):
    """Evaluate model performance using cross-validation with multiple metrics."""
    try:
        model = torch.load(model_path, map_location=device, weights_only=False)
        model.to(device)
        
        metrics = {
            'accuracy': [],
            'precision': [],
            'recall': [],
            'f1': []
        }
        
        kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
        
        for train_idx, val_idx in kf.split(X_data, y_data):
            X_train, X_val = X_data[train_idx], X_data[val_idx]
            y_train, y_val = y_data[train_idx], y_data[val_idx]
            
            X_val_t = torch.FloatTensor(X_val).to(device)
            y_val_np = y_val
            
            model.eval()
            with torch.no_grad():
                outputs = model(X_val_t).squeeze()
                if outputs.dim() == 1:
                    y_pred = (torch.sigmoid(outputs) > 0.5).long().cpu().numpy()
                else: 
                    y_pred = torch.softmax(outputs, dim=1).argmax(dim=1).cpu().numpy()
            
            metrics['accuracy'].append(accuracy_score(y_val_np, y_pred))
            metrics['precision'].append(precision_score(y_val_np, y_pred, average='weighted', zero_division=0))
            metrics['recall'].append(recall_score(y_val_np, y_pred, average='weighted', zero_division=0))
            metrics['f1'].append(f1_score(y_val_np, y_pred, average='weighted', zero_division=0))
        
        mean_metrics = {key: np.mean(values) for key, values in metrics.items()}
        std_metrics = {key: np.std(values) for key, values in metrics.items()}
        
        return mean_metrics, std_metrics
    
    except Exception as e:
        print(f"Error loading model {model_path.name}: {e}")
        return None, None


In [6]:
"""
Evaluating all models with comprehensive metrics.
"""

print("Evaluating all models with comprehensive metrics...")
print("This may take a few minutes...\n")

evaluation_results = []

for model_info in tqdm(all_models, desc="Evaluating Models"):
    model_name = model_info['file_path'].name
    
    mean_metrics, std_metrics = evaluate_model_performance(
        model_info['file_path'], 
        X_tensor.numpy(), 
        y_encoded,
        n_folds=3 
    )
    
    if mean_metrics is not None:
        result = {
            'model_name': model_name,
            'architecture': model_info['architecture'],
            'learning_rate': model_info['learning_rate'],
            'activation': model_info['activation'],
            'accuracy': mean_metrics['accuracy'],
            'precision': mean_metrics['precision'],
            'recall': mean_metrics['recall'],
            'f1': mean_metrics['f1'],
            'accuracy_std': std_metrics['accuracy'],
            'f1_std': std_metrics['f1']
        }
        
        evaluation_results.append(result)
    else:
        print(f"Skipping {model_name} due to evaluation error")


Evaluating all models with comprehensive metrics...
This may take a few minutes...



Evaluating Models: 100%|██████████| 2/2 [00:00<00:00, 1139.14it/s]

Error loading model modelo_0.7_8-10.pth: Can't get attribute 'MLPModel' on <module '__main__'>
Skipping modelo_0.7_8-10.pth due to evaluation error
Error loading model modelo_0.68_8-10.pth: Can't get attribute 'MLPModel' on <module '__main__'>
Skipping modelo_0.68_8-10.pth due to evaluation error


In [7]:
"""
Convert to DataFrame for easier analysis
"""

results_df = pd.DataFrame(evaluation_results)
print(f"\nSuccessfully evaluated {len(results_df)} models")

if len(results_df) > 0:
    print("\nTop 3 models by Accuracy:")
    print(results_df.nlargest(3, 'accuracy')[['model_name', 'accuracy', 'precision', 'recall', 'f1']])
    
    print("\nTop 3 models by F1-Score:")
    print(results_df.nlargest(3, 'f1')[['model_name', 'f1', 'accuracy', 'precision', 'recall']])



Successfully evaluated 0 models
